
# Diachronic Sentiment Arcs — 2025 Teaching Notebook (Clean & Reproducible)

**Audience:** non‑STEM students and newcomers to text mining  
**Goal:** turn long text into a *time series of affect* (“sentiment arc”), compare three sentiment families, smooth responsibly (moving avg / EMA / **Savitzky–Golay**), detect peaks/valleys, and discuss uncertainty.

### Reading the roadmap
1. Setup (reproducible environment + versions)  
2. Load or paste text → sentence segmentation  
3. Sentiment families: **Lexicon (VADER)**, **Classic ML baseline**, **Transformers (DistilBERT)**  
4. From sentences → time series (scores and confidence)  
5. Smoothing: MA, EMA, **Savitzky–Golay** (when/why)  
6. Peaks/valleys + interpretation caveats  
7. Robustness: model agreement, bootstrap bands  
8. Ethics & limitations  
9. **R Appendix** (optional): syuzhet/sentimentr, if your runtime supports R/rpy2

> **Pedagogical reminder:** these curves are *descriptive*. Peaks suggest clusters of positive sentences; valleys, negative. They are not causal proof of emotions or author intent.



## 1) Setup & Environment

- Run the install cell **once at the top** (avoids state drift).
- Then capture versions for reproducibility.


In [ ]:

%%bash
pip -q install "transformers>=4.44,<5" "datasets>=3.0,<3.1" "accelerate>=1.0" \
                "vaderSentiment>=3.3.2" "textblob>=0.18" "matplotlib>=3.9" \
                "plotly>=5.24" "pysbd>=0.3" "scipy>=1.13" "tqdm>=4.66" \
                "scikit-learn>=1.5"
python - <<'PY'
import nltk
nltk.download("punkt", quiet=True)
nltk.download("vader_lexicon", quiet=True)
PY


In [ ]:

# Capture versions (keep outputs in your notebook for reproducibility)
import sys, platform, importlib
print("Python:", sys.version)
for mod in ["torch","transformers","datasets","pandas","numpy","matplotlib","scipy","sklearn"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod}:", getattr(m, "__version__", "n/a"))
    except Exception as e:
        print(f"{mod}: not installed ({e})")



## 2) Configuration

Centralized parameters so we can tweak behavior without editing many cells.


In [ ]:

from dataclasses import dataclass
from pathlib import Path
import numpy as np, random
import re

@dataclass
class Config:
    project_name: str = "SentimentArcs"
    seed: int = 7
    # Transformer classifier (binary SST-2 is good for pedagogy)
    model_name: str = "distilbert-base-uncased-finetuned-sst-2-english"
    batch_size: int = 32
    max_length: int = 256
    # Smoothing choices
    smoothing: str = "savgol"    # options: "ma","ema","savgol"
    window_frac: float = 0.10    # ~10% of series length
    savgol_poly: int = 3         # poly order for Savitzky–Golay
    ema_alpha: float = 0.2       # EMA responsiveness (0<alpha<=1)
    # Peak detection
    peak_min_frac: float = 0.05  # min distance as % of series length
    # Plotting
    interactive: bool = True     # also render Plotly charts

CFG = Config()

# Seeding
np.random.seed(CFG.seed)
random.seed(CFG.seed)
try:
    import torch
    torch.manual_seed(CFG.seed)
except Exception:
    pass

out_dir = Path("./outputs"); out_dir.mkdir(exist_ok=True, parents=True)



## 3) Load or Paste Your Text → Sentence Segmentation

Paste text in the cell below **or** mount Drive and read from a file. We prefer clear, idempotent functions for cleaning/segmenting.


In [ ]:

SAMPLE_TEXT = '''
Alice felt cheerful on Monday. The sun was bright, and the city buzzed.
On Tuesday, a storm hit; plans were canceled, and frustration rose.
Wednesday brought relief as friends visited. Thursday was neutral.
By Friday, optimism returned in full force!
'''


In [ ]:

def clean_text(txt: str) -> str:
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def segment_sentences(txt: str):
    try:
        import pysbd
        seg = pysbd.Segmenter(language="en", clean=True)
        sents = [s.strip() for s in seg.segment(txt)]
    except Exception:
        # very simple fallback: split on punctuation
        sents = re.split(r"(?<=[.!?])\s+", txt)
    return [s for s in sents if s]


In [ ]:

import pandas as pd
text = clean_text(SAMPLE_TEXT)
sentences = segment_sentences(text)
df = pd.DataFrame({"i": range(len(sentences)), "sentence": sentences})
df.head()



## 4) Three Sentiment Families (Compare & Contrast)

- **Lexicon (VADER)**: rules + valence shifters. Fast, transparent. Domain slang may be tricky.  
- **Classic ML baseline (TF‑IDF + Logistic Regression)**: requires labels to train; we include a scaffold.  
- **Transformers (DistilBERT)**: contextual; strong accuracy; heavier. We use `pipeline(..., batch_size=CFG.batch_size)` with explicit model choice to keep results stable.


### 4.1 Lexicon: VADER (no training needed)

In [ ]:

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def vader_score(sent: str) -> float:
    # return compound in [-1,1]
    return float(sia.polarity_scores(sent)["compound"])

df["vader"] = [vader_score(s) for s in df["sentence"]]
df.head()



### 4.2 Classic ML Baseline (optional, needs labels)

If you have labeled data, train TF‑IDF + Logistic Regression. Here we only show a scaffold.


In [ ]:

# Scaffold (no training without labels)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(
    TfidfVectorizer(max_features=20000, ngram_range=(1,2)),
    LogisticRegression(max_iter=1000)
)
# Example (skip actual fit):
# baseline.fit(train_texts, train_labels)
# proba = baseline.predict_proba(df["sentence"])[:, 1]  # map to [-1,1] later


### 4.3 Transformers: DistilBERT sentiment (batched & explicit)

In [ ]:

from transformers import pipeline

clf = pipeline(
    task="sentiment-analysis",
    model=CFG.model_name,
    tokenizer=CFG.model_name,
    truncation=True,
    max_length=CFG.max_length,
    device_map="auto"   # will use GPU if available
)

# batched inference
preds = clf(df["sentence"].tolist(), batch_size=CFG.batch_size, padding=True, truncation=True)
# Convert labels to signed scores multiplied by confidence
label_to_sign = {"NEGATIVE": -1, "POSITIVE": +1, "NEUTRAL": 0}
def signed(p):
    return label_to_sign.get(p["label"].upper(), 0) * float(p.get("score", 1.0))

df["hf_score"] = [signed(p) for p in preds]
df.head()



## 5) From Sentences → Time Series + Smoothing

We get one score per sentence, then smooth.  
**Smoothing choices:** Moving Average (`window`), Exponential MA (`alpha`), **Savitzky–Golay** (`window_length` *odd*, `polyorder < window_length`).  
*Rule of thumb:* start with a window covering **5–15%** of points.


In [ ]:

import numpy as np
def z(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    return (x - np.nanmean(x)) / (np.nanstd(x) + 1e-8)

def moving_average(x, k):
    k = max(1, int(k))
    return np.convolve(x, np.ones(k) / k, mode="same")

def exponential_moving_average(x, alpha):
    y = np.zeros_like(x, dtype=float)
    y[0] = x[0]
    for t in range(1, len(x)):
        y[t] = alpha * x[t] + (1 - alpha) * y[t-1]
    return y

def savgol_smooth(x, window_length, polyorder):
    try:
        from scipy.signal import savgol_filter
        wl = max(3, int(window_length) | 1)      # ensure odd & >=3
        poly = min(polyorder, wl-1)
        return savgol_filter(x, wl, poly, mode="interp")
    except Exception:
        # fallback to moving average
        return moving_average(x, max(3, int(window_length)))

def smooth_series(y, method="savgol", window_frac=0.1, poly=3, alpha=0.2):
    y = np.asarray(y, float)
    n = len(y)
    k = max(3, int(np.ceil(n*window_frac)) | 1)  # odd
    if method == "ma":
        return moving_average(y, k)
    if method == "ema":
        return exponential_moving_average(y, alpha)
    return savgol_smooth(y, k, poly)

# Prepare series
df["vader_z"] = z(df["vader"].values)
df["hf_z"] = z(df["hf_score"].values)
df["mean_z"] = z(np.nanmean(np.vstack([df["vader_z"], df["hf_z"]]), axis=0))

df["mean_z_smooth"] = smooth_series(df["mean_z"].values, method=CFG.smoothing,
                                    window_frac=CFG.window_frac, poly=CFG.savgol_poly,
                                    alpha=CFG.ema_alpha)
df[["i","sentence","vader_z","hf_z","mean_z","mean_z_smooth"]].head()



## 6) Visualizing the Arc (static + optional interactive)

- **Static** first for quick reading.  
- **Interactive** (Plotly) adds hover text and easier peak inspection.


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(df["i"], df["mean_z"], label="mean z", linewidth=1)
ax.plot(df["i"], df["mean_z_smooth"], label=f"smooth ({CFG.smoothing})", linewidth=2)
ax.set_xlabel("Sentence #"); ax.set_ylabel("Sentiment (z-score)")
ax.set_title("Diachronic Sentiment Arc")
ax.legend(loc="best")
plt.show()


In [ ]:

if CFG.interactive:
    import plotly.graph_objects as go
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["i"], y=df["mean_z"], mode="lines", name="mean z",
                             hovertext=df["sentence"], hoverinfo="text+x+y"))
    fig.add_trace(go.Scatter(x=df["i"], y=df["mean_z_smooth"], mode="lines", name=f"smooth ({CFG.smoothing})"))
    fig.update_layout(title="Diachronic Sentiment Arc (Interactive)",
                      xaxis_title="Sentence #", yaxis_title="Sentiment (z-score)")
    fig.show()



## 7) Peaks/Valleys + Uncertainty

- We detect peaks on the **smoothed** series with a minimum spacing (e.g., 5% of the length).  
- We bootstrap (resample sentences with replacement) to estimate an uncertainty band (±1 SD) around the curve.


In [ ]:

from math import ceil
try:
    from scipy.signal import find_peaks
except Exception:
    def find_peaks(x, distance=1):
        # very simple fallback: local maxima
        x = np.asarray(x)
        peaks = [i for i in range(1, len(x)-1) if x[i]>x[i-1] and x[i]>x[i+1]]
        # thin by distance
        out = []
        for p in peaks:
            if not out or p - out[-1] >= distance: out.append(p)
        return np.array(out), {}
dist = max(1, int(ceil(len(df)*CFG.peak_min_frac)))
peaks, _ = find_peaks(df["mean_z_smooth"].values, distance=dist)
df_peaks = df.loc[peaks, ["i","sentence","mean_z_smooth"]]
df_peaks


In [ ]:

# Bootstrap uncertainty band (quick & light for teaching)
import numpy as np
B = 200  # keep small for class runtimes
curves = []
vals = df["mean_z"].values
for b in range(B):
    idx = np.random.RandomState(CFG.seed+b).choice(len(vals), size=len(vals), replace=True)
    series = vals[idx]
    sm = smooth_series(series, method=CFG.smoothing, window_frac=CFG.window_frac,
                       poly=CFG.savgol_poly, alpha=CFG.ema_alpha)
    curves.append(sm)
curves = np.vstack(curves)
mean = curves.mean(axis=0)
sd = curves.std(axis=0)
df["smooth_mean"] = mean
df["smooth_sd"] = sd


In [ ]:

import matplotlib.pyplot as plt
x = df["i"].values
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(x, df["mean_z_smooth"], label="smooth", linewidth=2)
ax.fill_between(x, df["smooth_mean"]-df["smooth_sd"], df["smooth_mean"]+df["smooth_sd"],
                alpha=0.2, label="bootstrap ±1 SD")
ax.scatter(df_peaks["i"], df_peaks["mean_z_smooth"], marker="x", s=60, label="peaks")
ax.set_title("Uncertainty Band & Peaks")
ax.set_xlabel("Sentence #"); ax.set_ylabel("Sentiment (z-score)")
ax.legend(loc="best")
plt.show()



## 8) Model Agreement (Correlation) & Neutral Share

We compare normalized series and report simple statistics to caution against over‑interpreting noisy arcs.


In [ ]:

corr = df[["vader_z","hf_z","mean_z"]].corr(method="spearman")
display(corr)
neutral_share = float((np.abs(df["mean_z"]) < 0.25).mean())
print("Neutral-ish share (|z|<0.25):", round(neutral_share, 3))



## 9) Ethics & Limitations

- **Domain shift:** models trained on movie/product reviews may misread literature, news, or historical texts.  
- **Irony/sarcasm:** still hard for machines.  
- **Cultural nuance:** lexicons and labels embed biases.  
- **Over-smoothing:** can erase minority voices/events; **under-smoothing** overreacts to single sentences.  
- **Do not claim causality** from curve shapes.



---
# R Appendix (Optional)

Use only if your environment supports `rpy2` and R packages; otherwise skip.


In [ ]:

USE_R = False  # set True if you have R + rpy2 + packages installed


In [ ]:

if USE_R:
    # In Colab you can enable R magic by installing rpy2, then:
    # %load_ext rpy2.ipython
    # %%R
    # install.packages(c("sentimentr","syuzhet"))
    # library(sentimentr); library(syuzhet)
    # # ... run R-based arcs here ...
    pass



**What these R packages do (theory in 2 lines):**  
- `sentimentr`: sentence-level valence with valence shifters (similar aim to VADER).  
- `syuzhet`: NRC emotion lexicon + smoothing method for “emotional arcs.”  
We included Python analogues so the main notebook stays self‑contained.
